# 47. Minimal API serving and monitoring
A prediction API receives structured input, validates it, calls a trusted fitted workflow, and returns structured output. A static course page does not host a Python service. This lab is a local demonstration; production deployment adds authentication, capacity, dependency, logging, and operational requirements.

Create a minimal FastAPI endpoint with a Pydantic request schema. Load the pipeline from a trusted artifact at startup for a real service. Reject invalid requests clearly and keep units explicit. Monitor input distributions, missingness, latency, and failure rates. Input drift is a change in inputs; it does not by itself prove accuracy fell. Performance monitoring needs representative outcomes when they arrive.

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field
from sklearn.linear_model import LinearRegression
model = LinearRegression().fit([[1], [2], [3]], [3, 5, 7]) # A tiny demonstration estimator.
app = FastAPI() # Create a local API application.
class Request(BaseModel):
    hours: float = Field(ge=0, le=24) # Require a bounded numerical input.
@app.post("/predict")
def predict(request: Request):
    return {"prediction": float(model.predict([[request.hours]])[0])} # Return JSON-compatible output.
client = TestClient(app) # Exercise the local endpoint without a live server.
response = client.post("/predict", json={"hours": 4}) # Send valid input.
print("Valid:", response.status_code, response.json()) # Inspect HTTP status and result.
print("Invalid status:", client.post("/predict", json={"hours": -1}).status_code) # Demonstrate schema rejection.
assert response.status_code == 200 # Verify successful local serving behavior.

## Monitor inputs without pretending you know accuracy
Monitoring has several layers: schema and range violations, missingness, latency, failure rates, input-distribution changes, and outcome-based quality. A two-sample Kolmogorov–Smirnov statistic measures the largest gap between empirical cumulative distributions of one numerical feature. Its p-value depends on sample size and assumptions, so a tiny p-value alone is not an operational alert policy. Repeated testing across many features increases false alarms. Define windows, sample counts, practical effect thresholds, seasonal references and investigation steps before observing an incident. Changes in y given X can occur with little visible input drift, and input drift can occur without worse predictions.

In [ ]:
import numpy as np
from scipy.stats import ks_2samp
rng = np.random.default_rng(42)
reference = rng.normal(0, 1, 200) # A retained training-era reference window.
current = rng.normal(0.6, 1, 200) # An illustrative shifted input window.
result = ks_2samp(reference, current) # Compare one numerical marginal.
print("Window sizes:", len(reference), len(current))
print("KS effect statistic:", round(result.statistic, 3))
print("KS p-value:", round(result.pvalue, 4))
print("Missing rate:", round(np.isnan(current).mean(), 3))
assert 0 <= result.statistic <= 1

### Reading the result
The statistic describes an input change, not predictive degradation. No target outcomes were supplied, so no accuracy claim is possible. A practical monitor logs version, timestamps, counts, schema-failure rate, latency and the outcome-delay policy. Compare labeled predictions later using representative labels and subgroup checks.

### Transfer exercise
Write a monitor specification for an earlier project: reference window, current window, minimum sample size, practical alert rule, responsible action and outcome-based follow-up.

## Your turn
Build a local prediction endpoint for an earlier saved pipeline. Define three monitoring indicators and explain which require true outcomes.

Try the task before opening the solution on the course website.